# C-MAPSS Remaining Useful Life Prediction with a Deep Convolutional Neural Network

This notebook trains a DCNN to predict Remaining Useful Life (RUL) from the NASA C-MAPSS turbofan engine data using a sliding time-window approach, then turns the predicted RUL into a simple maintenance recommendation.

It is designed for `FD001` to `FD004` and can be run on one subset or loop over all four subsets.

## 1) Imports and configuration
Change `DATA_DIR` to the folder that contains the files:
`train_FD001.txt`, `test_FD001.txt`, `RUL_FD001.txt`, ... `FD004`.

In [41]:
from __future__ import annotations

from pathlib import Path
from typing import List
import math
import random
import warnings
import json

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

# -----------------------------
# Data / experiment config
# -----------------------------
DATA_DIR = Path("Data")  # <-- change this
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

SEQ_LEN = 30
RUL_CEILING = 125       # common practice in C-MAPSS regression
BATCH_SIZE = 64
EPOCHS = 150
LR = 5e-4
PATIENCE = 20
VAL_SIZE = 0.2
NUM_WORKERS = 0

Using device: mps


## 2) Load the NASA C-MAPSS files
The training files contain cycle-by-cycle sensor snapshots, and the test sets come with a separate `RUL_*.txt` file that gives the remaining life at the final observed cycle for each engine unit.

In [42]:
COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]


def load_subset(data_dir, subset):
    train_path = data_dir / f"train_{subset}.txt"
    test_path  = data_dir / f"test_{subset}.txt"
    rul_path   = data_dir / f"RUL_{subset}.txt"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing file: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing file: {test_path}")
    if not rul_path.exists():
        raise FileNotFoundError(f"Missing file: {rul_path}")

    train_df = pd.read_csv(train_path, sep=r"\s+", header=None)
    test_df  = pd.read_csv(test_path,  sep=r"\s+", header=None)
    rul = pd.read_csv(rul_path, sep=r"\s+", header=None).iloc[:, 0].astype(int).to_numpy()

    train_df.columns = COLS
    test_df.columns  = COLS
    return train_df, test_df, rul


def add_rul_to_train(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    max_cycle = out.groupby("unit")["cycle"].transform("max")
    out["RUL"] = (max_cycle - out["cycle"]).clip(upper=RUL_CEILING)
    return out


def add_rul_to_test(df: pd.DataFrame, end_rul) -> pd.DataFrame:
    """
    For each test unit:
      RUL at a row = end_rul_from_file + (max_cycle_of_unit - current_cycle)
    Then clip to RUL_CEILING.
    """
    out = df.copy()
    unit_ids = sorted(out["unit"].unique())
    unit_to_end_rul = {u: int(r) for u, r in zip(unit_ids, end_rul)}

    max_cycle = out.groupby("unit")["cycle"].transform("max")
    out["end_rul"] = out["unit"].map(unit_to_end_rul)
    out["RUL"] = (out["end_rul"] + (max_cycle - out["cycle"])).clip(upper=RUL_CEILING)
    return out


In [43]:
train_df, test_df, rul = load_subset(DATA_DIR, SUBSETS[0])
train_df = add_rul_to_train(train_df)
test_df = add_rul_to_test(test_df, rul)

In [31]:
display(train_df.head(5), test_df.head(5), train_df.shape, test_df.shape)

,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,-1.645790,0.197786,-0.125176,-0.624156,-1.182131,-1.339935,-1.077492,-0.970956,...,0.193546,-0.774069,-0.519706,0.685019,1.239046,0.869143,-0.726712,0.639215,0.623141,125
1,1,2,2.031488,1.088098,-0.238302,-0.837462,-0.635382,-0.340452,-1.155714,-0.572142,...,0.721452,-0.719653,-1.306246,2.335509,0.697480,-1.149332,0.432017,0.767292,-2.751969,125
2,1,3,-0.563649,1.207723,0.316027,0.324504,0.274585,-0.872209,-0.246266,-0.978345,...,-0.982591,1.190514,-1.640184,-0.134296,-0.141040,-0.333887,-0.208700,-1.604032,2.031152,125
3,1,4,-0.935090,0.186765,-0.391524,0.339533,-0.432154,-2.197296,-0.260709,-0.645098,...,-1.517097,0.636501,-2.099022,0.147575,1.292158,-0.060814,0.868521,-0.544085,-0.550668,125
4,1,5,-1.016924,0.076929,0.894514,2.928866,-2.061116,-2.790145,0.652766,0.328249,...,2.136142,-1.339476,-0.103365,1.346357,-0.808031,0.265084,2.600511,1.037941,-0.304593,125


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,end_rul,RUL
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735,112,125
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916,112,125
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166,112,125
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737,112,125
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130,112,125


(6962, 27)

(13096, 28)

## 3) Preprocessing
We automatically remove constant sensors from the training data, scale the remaining features using the training set only, and cap the target RUL at `RUL_CEILING`.

In [32]:
def select_features(train_df: pd.DataFrame):
    """Drop zero- or near-zero-variance columns (constant sensors)."""
    op_and_sensor_cols = [c for c in COLS[2:] if c in train_df.columns]
    feature_cols = [c for c in op_and_sensor_cols if train_df[c].std() > 0]
    return feature_cols


def preprocess_subset(train_df: pd.DataFrame, test_df: pd.DataFrame, end_rul):
    train_df = add_rul_to_train(train_df)
    test_df  = add_rul_to_test(test_df, end_rul)

    feature_cols = select_features(train_df)

    # Train data is already z-scored (N≈0, std≈1).
    # Test data is raw — fit a separate scaler on test features only,
    # normalising it to the same N(0,1) scale that the model was trained on.
    # Doing so aligns the BatchNorm running statistics with what the model sees
    # during inference, avoiding the scale-explosion seen with a combined fit.
    scaler = StandardScaler()
    test_arr = scaler.fit_transform(test_df[feature_cols])

    train_df = train_df.copy()
    test_df  = test_df.copy()
    # Train features are left as-is (already normalised by the data provider).
    train_df[feature_cols] = np.nan_to_num(train_df[feature_cols].values, nan=0.0, posinf=0.0, neginf=0.0)
    test_df[feature_cols]  = np.nan_to_num(test_arr,                       nan=0.0, posinf=0.0, neginf=0.0)

    # RUL is already clipped in add_rul_to_train/add_rul_to_test
    train_df["RUL_CAPPED"] = train_df["RUL"]
    test_df["RUL_CAPPED"]  = test_df["RUL"]

    return train_df, test_df, feature_cols, scaler


## 4) Build sliding windows
Each sample is a sequence of the last `SEQ_LEN` cycles. The model predicts the RUL at the last cycle of that window.

In [33]:
def make_windows(df: pd.DataFrame, feature_cols: List[str], target_col: str = 'RUL_CAPPED', seq_len: int = 30):
    X, y, groups, meta = [], [], [], []

    for unit_id, g in df.groupby('unit'):
        g = g.sort_values('cycle').reset_index(drop=True)
        features = g[feature_cols].to_numpy(dtype=np.float32)
        target = g[target_col].to_numpy(dtype=np.float32)
        cycles = g['cycle'].to_numpy(dtype=np.int32)

        if len(g) < seq_len:
            continue

        for i in range(seq_len - 1, len(g)):
            X.append(features[i - seq_len + 1:i + 1])
            y.append(target[i])
            groups.append(unit_id)
            meta.append((int(unit_id), int(cycles[i])))

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    groups = np.asarray(groups)
    meta = np.asarray(meta)

    return X, y, groups, meta


class RULWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 5) DCNN model
The architecture is a 1D convolutional regressor suitable for time-window sensor sequences.

In [34]:
class DCNNRegressor(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.backbone = nn.Sequential(
            # Block 1
            nn.Conv1d(n_features, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.3),
            # Block 2
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.3),
            # Block 3
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),   # more stable than MaxPool for regression
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),         # NO activation — raw regression output
        )

    def forward(self, x):
        # x: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.transpose(1, 2)
        x = self.backbone(x)
        x = self.head(x)
        return x


def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))


## 6) Training helpers
We split the windows by engine unit, not by random rows, so the validation set is genuinely unseen.

In [35]:
def fit_model(model, train_loader, val_loader, epochs=50, lr=1e-3, patience=8):
    criterion = nn.MSELoss()
    # AdamW for better weight decay handling than plain Adam
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
    )

    best_val   = float('inf')
    best_state = None
    wait       = 0
    history    = {'train_loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss   = float(np.mean(val_losses))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        print(f'Epoch {epoch:02d} | train MSE={train_loss:.4f} | val MSE={val_loss:.4f}')

        if val_loss < best_val - 1e-5:
            best_val   = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait       = 0
        else:
            wait += 1
            if wait >= patience:
                print('Early stopping triggered.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    trues = []
    for xb, yb in loader:
        xb   = xb.to(DEVICE)
        pred = model(xb).cpu().numpy().ravel()
        preds.extend(pred.tolist())
        trues.extend(yb.numpy().ravel().tolist())

    # Clamp negative predictions (RUL cannot be < 0)
    preds = [max(0.0, float(p)) for p in preds]

    print("y_pred range:", np.array(preds).min(), np.array(preds).max())
    print("y_true range:", np.array(trues).min(), np.array(trues).max())
    print("NaN in preds:", np.isnan(np.array(preds)).any())

    return np.array(trues), np.nan_to_num(np.array(preds), nan=0.0, posinf=0.0, neginf=0.0)


## 7) One complete experiment for a subset
This function trains one model, evaluates it, and returns both the trained model and the test predictions.

In [36]:
def run_experiment(subset: str, data_dir: Path = DATA_DIR):
    print(f"\n=== Running experiment: {subset} ===")
    train_df, test_df, end_rul = load_subset(data_dir, subset)
    train_df, test_df, feature_cols, scaler = preprocess_subset(train_df, test_df, end_rul)

    # Sliding windows
    X_train, y_train, groups_train, _         = make_windows(train_df, feature_cols, seq_len=SEQ_LEN)
    X_test,  y_test,  groups_test,  meta_test = make_windows(test_df,  feature_cols, seq_len=SEQ_LEN)
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

    # Sanity check shapes and RUL range
    print(f"X_train shape: {X_train.shape}")
    print(f"Train RUL range: {y_train.min():.1f} – {y_train.max():.1f}")
    print(f"Test  RUL range: {y_test.min():.1f} – {y_test.max():.1f}")

    # Group split by engine unit
    splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
    tr_idx, va_idx = next(splitter.split(X_train, y_train, groups=groups_train))

    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[va_idx], y_train[va_idx]

    train_loader = DataLoader(RULWindowDataset(X_tr, y_tr),    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = DataLoader(RULWindowDataset(X_va, y_va),    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(RULWindowDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = DCNNRegressor(n_features=len(feature_cols)).to(DEVICE)
    model, history = fit_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE)

    y_true_test, y_pred_test = predict(model, test_loader)
    test_rmse = rmse(y_true_test, y_pred_test)
    test_mae  = mean_absolute_error(y_true_test, y_pred_test)

    print(f'Test RMSE ({subset}): {test_rmse:.3f}')
    print(f'Test MAE  ({subset}): {test_mae:.3f}')

    results = pd.DataFrame({
        'unit':     meta_test[:, 0].astype(int),
        'cycle':    meta_test[:, 1].astype(int),
        'true_rul': y_true_test,
        'pred_rul': y_pred_test,
    })
    results['pred_rul_clipped'] = results['pred_rul'].clip(lower=0)

    return {
        'subset':       subset,
        'model':        model,
        'feature_cols': feature_cols,
        'scaler':       scaler,
        'history':      history,
        'test_results': results,
        'test_rmse':    test_rmse,
        'test_mae':     test_mae,
    }


## 8) Maintenance recommendation
A simple rule-based policy can turn the predicted RUL into an action.
Adjust the thresholds to match your operations policy.

In [37]:
def maintenance_recommendation(pred_rul: float, warning: int = 50, critical: int = 20) -> str:
    if pred_rul <= critical:
        return 'MAINTENANCE NOW'
    elif pred_rul <= warning:
        return 'Schedule maintenance soon'
    else:
        return 'Continue monitoring'


def add_recommendations(pred_df: pd.DataFrame, warning: int = 50, critical: int = 20) -> pd.DataFrame:
    out = pred_df.copy()
    out['recommendation'] = out['pred_rul_clipped'].apply(lambda x: maintenance_recommendation(x, warning, critical))
    out['maintenance_in_cycles'] = (out['pred_rul_clipped'] - critical).clip(lower=0)
    return out

## 9) Run on all four subsets
This will train one DCNN per subset. That is usually better than mixing them, because the FD subsets have different operating-condition / fault-mode settings.

In [38]:
all_outputs = {}

for subset in SUBSETS:
    out = run_experiment(subset, DATA_DIR)
    all_outputs[subset] = out

summary = pd.DataFrame([
    {
        'subset': s,
        'test_rmse': all_outputs[s]['test_rmse'],
        'test_mae': all_outputs[s]['test_mae'],
        'n_test_windows': len(all_outputs[s]['test_results']),
    }
    for s in SUBSETS
])

summary


=== Running experiment: FD001 ===
X_train shape: (5512, 30, 24)
Train RUL range: 0.0 – 125.0
Test  RUL range: 7.0 – 125.0
Epoch 01 | train MSE=4090.1171 | val MSE=3941.1390
Epoch 02 | train MSE=1903.2033 | val MSE=4741.4081
Epoch 03 | train MSE=613.4493 | val MSE=4461.2408
Epoch 04 | train MSE=391.3041 | val MSE=4049.8192
Epoch 05 | train MSE=350.6008 | val MSE=3423.6726
Epoch 06 | train MSE=314.7590 | val MSE=4049.9112
Epoch 07 | train MSE=264.7186 | val MSE=3971.8288
Epoch 08 | train MSE=268.3391 | val MSE=3607.1959
Epoch 09 | train MSE=251.3131 | val MSE=3552.1970
Epoch 10 | train MSE=239.7133 | val MSE=3588.0480
Epoch 11 | train MSE=229.4479 | val MSE=3488.1280
Epoch 12 | train MSE=226.9023 | val MSE=3584.7057
Epoch 13 | train MSE=218.0986 | val MSE=3543.3042
Epoch 14 | train MSE=217.9689 | val MSE=3626.1501
Epoch 15 | train MSE=206.9577 | val MSE=3601.3375
Epoch 16 | train MSE=219.1886 | val MSE=3493.1000
Epoch 17 | train MSE=204.5724 | val MSE=3569.8377
Epoch 18 | train MSE=205.

,subset,test_rmse,test_mae,n_test_windows
0,FD001,83.432227,77.317667,10196
1,FD002,90.611660,83.416899,1611
2,FD003,92.437540,86.285270,2104
3,FD004,93.938733,89.533828,1746


## 10) Inspect the predicted RUL and maintenance recommendation
This shows the latest prediction per engine unit in the test set.

In [39]:
subset = 'FD001'   # change if needed

pred_df = all_outputs[subset]['test_results']
latest_pred_per_unit = (
    pred_df.sort_values(['unit', 'cycle'])
           .groupby('unit', as_index=False)
           .tail(1)
           .reset_index(drop=True)
)

latest_pred_per_unit = add_recommendations(latest_pred_per_unit, warning=50, critical=20)
latest_pred_per_unit[['unit', 'cycle', 'true_rul', 'pred_rul_clipped', 'recommendation']].head(20)

,unit,cycle,true_rul,pred_rul_clipped,recommendation
0,1,31,112.0,31.918659,Schedule maintenance soon
1,2,49,98.0,9.898246,MAINTENANCE NOW
2,3,126,69.0,14.588527,MAINTENANCE NOW
3,4,106,82.0,5.651301,MAINTENANCE NOW
4,5,98,91.0,4.538156,MAINTENANCE NOW
5,6,105,93.0,2.929143,MAINTENANCE NOW
6,7,160,91.0,39.059128,Schedule maintenance soon
7,8,166,95.0,13.583293,MAINTENANCE NOW
8,9,55,111.0,3.279015,MAINTENANCE NOW
9,10,192,96.0,4.993193,MAINTENANCE NOW


## 11) Save the trained model
You can save the model state, scaler, and selected features for deployment in an API or a scheduling script.

In [40]:
def save_artifacts(output: dict, save_dir: Path):
    save_dir.mkdir(parents=True, exist_ok=True)

    torch.save(output['model'].state_dict(), save_dir / f"{output['subset']}_dcnn.pt")

    metadata = {
        'subset': output['subset'],
        'feature_cols': output['feature_cols'],
        'seq_len': SEQ_LEN,
        'rul_ceiling': RUL_CEILING,
    }
    (save_dir / f"{output['subset']}_metadata.json").write_text(json.dumps(metadata, indent=2))

    scaler_data = {
        'mean_': output['scaler'].mean_.tolist(),
        'scale_': output['scaler'].scale_.tolist(),
        'var_': output['scaler'].var_.tolist(),
        'n_features_in_': int(output['scaler'].n_features_in_),
    }
    (save_dir / f"{output['subset']}_scaler.json").write_text(json.dumps(scaler_data, indent=2))

save_dir = Path('saved_cmapss_models')
for s in SUBSETS:
    save_artifacts(all_outputs[s], save_dir)

print('Saved to:', save_dir.resolve())

Saved to: /Users/hrishavmajumder/Documents/MIOM_Project/saved_cmapss_models
